# Diffusion Models: DDPM and Stable Diffusion

**Module 13: Generative Models (Part 2)**  
**Objective**: Master diffusion models from fundamentals to Stable Diffusion

Diffusion Models:
- Forward process (adding noise)
- Reverse process (denoising)
- DDPM (Denoising Diffusion Probabilistic Models)
- DDIM (Denoising Diffusion Implicit Models)
- Stable Diffusion architecture
- Text-to-image generation
- ControlNet and LoRA

## What You'll Learn
1. Diffusion process mathematics
2. DDPM training and sampling
3. DDIM accelerated sampling
4. Stable Diffusion components
5. ControlNet for controlled generation
6. LoRA for efficient fine-tuning

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple, List
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)

## 1. Diffusion Process

**Intuition**: Gradually add noise to data, then learn to reverse the process

**Forward process** (fixed):
$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}x_{t-1}, \beta_t I)$$

**Reverse process** (learned):
$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t))$$

**Key insight**: With appropriate noise schedule, after T steps, $x_T \sim \mathcal{N}(0, I)$

In [ ]:
def visualize_diffusion_process():
    """Visualize forward and reverse diffusion process."""
    
    fig, axes = plt.subplots(2, 6, figsize=(18, 7))
    
    # Create a simple signal
    t = np.linspace(0, 2*np.pi, 100)
    original_signal = np.sin(3*t) + 0.5*np.cos(7*t)
    
    # Noise schedule (β)
    betas = np.linspace(0.0001, 0.02, 1000)
    alphas = 1 - betas
    alphas_cumprod = np.cumprod(alphas)
    
    # Forward process
    timesteps_forward = [0, 100, 200, 400, 700, 999]
    for idx, timestep in enumerate(timesteps_forward):
        ax = axes[0, idx]
        
        if timestep == 0:
            noisy_signal = original_signal
        else:
            # x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
            alpha_bar = alphas_cumprod[timestep]
            noise = np.random.randn(*original_signal.shape) * 0.5
            noisy_signal = np.sqrt(alpha_bar) * original_signal + np.sqrt(1 - alpha_bar) * noise
        
        ax.plot(t, noisy_signal, 'b-', linewidth=2)
        ax.set_title(f'Forward t={timestep}', fontsize=10, weight='bold')
        ax.set_ylim(-3, 3)
        ax.grid(True, alpha=0.3)
        ax.set_xticks([])
        ax.set_yticks([])
        
        if timestep == 999:
            ax.text(np.pi, -2.5, 'Pure Noise', ha='center', 
                   fontsize=9, color='red', weight='bold')
    
    # Reverse process
    timesteps_reverse = [999, 700, 400, 200, 100, 0]
    for idx, timestep in enumerate(timesteps_reverse):
        ax = axes[1, idx]
        
        if timestep == 999:
            denoised_signal = np.random.randn(*original_signal.shape) * 0.5
        else:
            # Simulated denoising (not actual diffusion, just for visualization)
            alpha_bar = alphas_cumprod[timestep]
            progress = 1 - (timestep / 1000)
            denoised_signal = progress * original_signal + (1 - progress) * np.random.randn(*original_signal.shape) * 0.5
        
        ax.plot(t, denoised_signal, 'r-', linewidth=2)
        ax.set_title(f'Reverse t={timestep}', fontsize=10, weight='bold')
        ax.set_ylim(-3, 3)
        ax.grid(True, alpha=0.3)
        ax.set_xticks([])
        ax.set_yticks([])
        
        if timestep == 0:
            ax.text(np.pi, -2.5, 'Clean Data', ha='center', 
                   fontsize=9, color='green', weight='bold')
    
    # Add arrows between rows
    fig.text(0.5, 0.52, '▼ Forward: Add Noise', ha='center', fontsize=12, 
            weight='bold', color='blue')
    fig.text(0.5, 0.48, '▲ Reverse: Denoise (Learn This!)', ha='center', fontsize=12, 
            weight='bold', color='red')
    
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.3)
    plt.show()
    
    print("Diffusion Process:")
    print("="*70)
    print("\nFORWARD PROCESS (Fixed):")
    print("  • Start with clean data x₀")
    print("  • Gradually add Gaussian noise over T timesteps")
    print("  • At each step: x_t = √(1-βₜ) x_{t-1} + √βₜ ε")
    print("  • After T steps: x_T ≈ N(0, I) (pure noise)")
    
    print("\nREVERSE PROCESS (Learned):")
    print("  • Start with pure noise x_T ~ N(0, I)")
    print("  • Learn to denoise step by step")
    print("  • Neural network predicts noise ε_θ(x_t, t)")
    print("  • After T steps: x₀ (clean data)")
    
    print("\nKEY INSIGHT:")
    print("  • Forward process is mathematically tractable")
    print("  • Can sample x_t directly from x₀ (no need for T steps)")
    print("  • Train network to reverse this process")

visualize_diffusion_process()

## 2. Noise Schedule

**β schedule** controls noise addition rate

Common schedules:
- **Linear**: β increases linearly
- **Cosine**: Smoother transition
- **Quadratic**: Faster initial noise

$$\alpha_t = 1 - \beta_t$$
$$\bar{\alpha}_t = \prod_{i=1}^t \alpha_i$$

In [ ]:
def get_noise_schedule(schedule: str = "linear", timesteps: int = 1000) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Get noise schedule for diffusion.
    
    Args:
        schedule: Type of schedule ("linear", "cosine", "quadratic")
        timesteps: Number of diffusion steps
        
    Returns:
        betas: (T,) noise schedule
        alphas: (T,) 1 - betas
        alphas_cumprod: (T,) cumulative product of alphas
    """
    if schedule == "linear":
        beta_start = 0.0001
        beta_end = 0.02
        betas = torch.linspace(beta_start, beta_end, timesteps)
        
    elif schedule == "cosine":
        s = 0.008
        steps = timesteps + 1
        x = torch.linspace(0, timesteps, steps)
        alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        betas = torch.clip(betas, 0.0001, 0.9999)
        
    elif schedule == "quadratic":
        beta_start = 0.0001
        beta_end = 0.02
        betas = torch.linspace(beta_start**0.5, beta_end**0.5, timesteps) ** 2
        
    else:
        raise ValueError(f"Unknown schedule: {schedule}")
    
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    
    return betas, alphas, alphas_cumprod

# Compare schedules
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

schedules = ["linear", "cosine", "quadratic"]
colors = ['blue', 'green', 'red']

# Plot betas
ax1 = axes[0, 0]
ax1.set_title('Noise Schedule (β)', fontsize=12, weight='bold')
ax1.set_xlabel('Timestep')
ax1.set_ylabel('β_t')

for schedule, color in zip(schedules, colors):
    betas, _, _ = get_noise_schedule(schedule, 1000)
    ax1.plot(betas.numpy(), label=schedule.capitalize(), color=color, linewidth=2)

ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot alphas_cumprod
ax2 = axes[0, 1]
ax2.set_title('Cumulative Product (ᾱ)', fontsize=12, weight='bold')
ax2.set_xlabel('Timestep')
ax2.set_ylabel('ᾱ_t')

for schedule, color in zip(schedules, colors):
    _, _, alphas_cumprod = get_noise_schedule(schedule, 1000)
    ax2.plot(alphas_cumprod.numpy(), label=schedule.capitalize(), color=color, linewidth=2)

ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot signal-to-noise ratio
ax3 = axes[1, 0]
ax3.set_title('Signal-to-Noise Ratio', fontsize=12, weight='bold')
ax3.set_xlabel('Timestep')
ax3.set_ylabel('SNR (dB)')

for schedule, color in zip(schedules, colors):
    _, _, alphas_cumprod = get_noise_schedule(schedule, 1000)
    snr = alphas_cumprod / (1 - alphas_cumprod)
    snr_db = 10 * torch.log10(snr + 1e-10)
    ax3.plot(snr_db.numpy(), label=schedule.capitalize(), color=color, linewidth=2)

ax3.legend()
ax3.grid(True, alpha=0.3)

# Comparison table
ax4 = axes[1, 1]
ax4.axis('off')
ax4.set_title('Schedule Comparison', fontsize=12, weight='bold')

table_data = [
    ["Schedule", "Start β", "End β", "Characteristics"],
    ["Linear", "0.0001", "0.02", "Simple, uniform noise"],
    ["Cosine", "~0.0001", "~0.02", "Smoother, better quality"],
    ["Quadratic", "0.0001", "0.02", "Faster initial noise"]
]

for i, row in enumerate(table_data):
    y_pos = 0.85 - i * 0.12
    for j, cell in enumerate(row):
        x_pos = 0.05 + j * 0.22
        weight = 'bold' if i == 0 else 'normal'
        ax4.text(x_pos, y_pos, cell, fontsize=9, weight=weight, 
                verticalalignment='center')
        
    if i == 0:
        ax4.axhline(y=y_pos-0.05, xmin=0.05, xmax=0.95, 
                   color='black', linewidth=1)

plt.tight_layout()
plt.show()

print("\nNoise Schedule Analysis:")
print("="*70)

for schedule in schedules:
    betas, alphas, alphas_cumprod = get_noise_schedule(schedule, 1000)
    
    print(f"\n{schedule.upper()}:")
    print(f"  β range: [{betas[0]:.6f}, {betas[-1]:.6f}]")
    print(f"  Final ᾱ: {alphas_cumprod[-1]:.6f}")
    print(f"  Signal retention at t=500: {alphas_cumprod[500]:.4f}")

## 3. DDPM: Denoising Diffusion Probabilistic Models

**Training objective**: Predict noise ε added to image

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon} \left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]$$

Where:
- $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon$
- $\epsilon_\theta$ is neural network that predicts noise

In [ ]:
class SinusoidalPositionEmbedding(nn.Module):
    """
    Sinusoidal position embedding for timestep encoding.
    """
    
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
    
    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        """
        Args:
            timesteps: (batch_size,)
            
        Returns:
            embeddings: (batch_size, dim)
        """
        device = timesteps.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = timesteps[:, None] * embeddings[None, :]
        embeddings = torch.cat([torch.sin(embeddings), torch.cos(embeddings)], dim=-1)
        return embeddings

class SimpleUNet(nn.Module):
    """
    Simplified U-Net for diffusion models.
    
    U-Net is commonly used because:
    - Skip connections preserve details
    - Multi-scale processing
    - Efficient for image-to-image tasks
    """
    
    def __init__(self, in_channels: int = 3, out_channels: int = 3, 
                 time_dim: int = 256, base_channels: int = 64):
        super().__init__()
        
        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim)
        )
        
        # Downsampling
        self.down1 = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 3, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.ReLU()
        )
        
        self.down2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels*2, 3, padding=1, stride=2),
            nn.GroupNorm(8, base_channels*2),
            nn.ReLU()
        )
        
        self.down3 = nn.Sequential(
            nn.Conv2d(base_channels*2, base_channels*4, 3, padding=1, stride=2),
            nn.GroupNorm(8, base_channels*4),
            nn.ReLU()
        )
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(base_channels*4, base_channels*4, 3, padding=1),
            nn.GroupNorm(8, base_channels*4),
            nn.ReLU()
        )
        
        # Upsampling
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(base_channels*4, base_channels*2, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_channels*2),
            nn.ReLU()
        )
        
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(base_channels*4, base_channels, 4, stride=2, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.ReLU()
        )
        
       self.up1 = nn.Sequential(
            nn.Conv2d(base_channels*2, base_channels, 3, padding=1),
            nn.GroupNorm(8, base_channels),
            nn.ReLU()
        )
        
        # Output
        self.output = nn.Conv2d(base_channels, out_channels, 1)
        
        # Time projection layers
        self.time_proj1 = nn.Linear(time_dim, base_channels)
        self.time_proj2 = nn.Linear(time_dim, base_channels*2)
        self.time_proj3 = nn.Linear(time_dim, base_channels*4)
        
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, channels, height, width) noisy image
            t: (batch,) timestep
            
        Returns:
            noise_pred: (batch, channels, height, width) predicted noise
        """
        # Time embedding
        t_emb = self.time_mlp(t)
        
        # Downsampling with skip connections
        x1 = self.down1(x)  # (B, 64, H, W)
        x1 = x1 + self.time_proj1(t_emb)[:, :, None, None]
        
        x2 = self.down2(x1)  # (B, 128, H/2, W/2)
        x2 = x2 + self.time_proj2(t_emb)[:, :, None, None]
        
        x3 = self.down3(x2)  # (B, 256, H/4, W/4)
        x3 = x3 + self.time_proj3(t_emb)[:, :, None, None]
        
        # Bottleneck
        x = self.bottleneck(x3)
        
        # Upsampling with skip connections
        x = self.up3(x)
        x = torch.cat([x, x2], dim=1)
        
        x = self.up2(x)
        x = torch.cat([x, x1], dim=1)
        
        x = self.up1(x)
        
        # Output
        return self.output(x)

class DDPM:
    """
    Denoising Diffusion Probabilistic Model.
    """
    
    def __init__(self, model: nn.Module, timesteps: int = 1000, 
                 schedule: str = "linear", device: str = "cpu"):
        self.model = model.to(device)
        self.timesteps = timesteps
        self.device = device
        
        # Get noise schedule
        betas, alphas, alphas_cumprod = get_noise_schedule(schedule, timesteps)
        
        self.betas = betas.to(device)
        self.alphas = alphas.to(device)
        self.alphas_cumprod = alphas_cumprod.to(device)
        
        # Precompute terms for training
        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod).to(device)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod).to(device)
        
    def q_sample(self, x_start: torch.Tensor, t: torch.Tensor, 
                 noise: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward diffusion: sample x_t from x_0.
        
        x_t = √ᾱ_t * x_0 + √(1-ᾱ_t) * ε
        
        Args:
            x_start: (B, C, H, W) clean images
            t: (B,) timesteps
            noise: (B, C, H, W) noise (optional)
            
        Returns:
            x_t: (B, C, H, W) noisy images
        """
        if noise is None:
            noise = torch.randn_like(x_start)
        
        sqrt_alpha_t = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_one_minus_alpha_t = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        
        return sqrt_alpha_t * x_start + sqrt_one_minus_alpha_t * noise
    
    def p_losses(self, x_start: torch.Tensor, t: torch.Tensor, 
                 noise: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Compute training loss.
        
        Loss = ||ε - ε_θ(x_t, t)||²
        
        Args:
            x_start: (B, C, H, W) clean images
            t: (B,) timesteps
            noise: (B, C, H, W) noise (optional)
            
        Returns:
            loss: scalar
        """
        if noise is None:
            noise = torch.randn_like(x_start)
        
        # Forward diffusion
        x_noisy = self.q_sample(x_start, t, noise)
        
        # Predict noise
        noise_pred = self.model(x_noisy, t)
        
        # MSE loss
        loss = F.mse_loss(noise, noise_pred)
        
        return loss
    
    @torch.no_grad()
    def p_sample(self, x: torch.Tensor, t: int) -> torch.Tensor:
        """
        Reverse diffusion: single step x_t → x_{t-1}.
        
        Args:
            x: (B, C, H, W) noisy images at step t
            t: timestep
            
        Returns:
            x_prev: (B, C, H, W) images at step t-1
        """
        batch_size = x.shape[0]
        t_tensor = torch.full((batch_size,), t, device=self.device, dtype=torch.long)
        
        # Predict noise
        noise_pred = self.model(x, t_tensor)
        
        # Get alpha, beta terms
        alpha_t = self.alphas[t]
        alpha_bar_t = self.alphas_cumprod[t]
        beta_t = self.betas[t]
        
        # Compute mean
        mean = (1 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * noise_pred
        )
        
        if t > 0:
            noise = torch.randn_like(x)
            variance = beta_t
            x_prev = mean + torch.sqrt(variance) * noise
        else:
            x_prev = mean
        
        return x_prev
    
    @torch.no_grad()
    def sample(self, shape: Tuple[int, ...]) -> torch.Tensor:
        """
        Generate samples from noise.
        
        Args:
            shape: (batch_size, channels, height, width)
            
        Returns:
            samples: (batch_size, channels, height, width)
        """
        self.model.eval()
        
        # Start from pure noise
        x = torch.randn(shape, device=self.device)
        
        # Reverse diffusion
        for t in reversed(range(self.timesteps)):
            x = self.p_sample(x, t)
        
        return x

# Create DDPM
unet = SimpleUNet(in_channels=3, out_channels=3, time_dim=256, base_channels=64)
ddpm = DDPM(unet, timesteps=1000, schedule="cosine", device=device)

total_params = sum(p.numel() for p in unet.parameters())

print("DDPM Architecture:")
print("="*70)
print(f"Total parameters: {total_params:,}")
print(f"Timesteps: 1000")
print(f"Schedule: cosine")

# Test forward pass
batch_size = 4
test_images = torch.randn(batch_size, 3, 32, 32).to(device)
test_timesteps = torch.randint(0, 1000, (batch_size,)).to(device)

with torch.no_grad():
    noise_pred = unet(test_images, test_timesteps)

print(f"\nForward pass:")
print(f"  Input: {test_images.shape}")
print(f"  Timesteps: {test_timesteps.shape}")
print(f"  Predicted noise: {noise_pred.shape}")

# Test training loss
loss = ddpm.p_losses(test_images, test_timesteps)
print(f"\nTraining loss: {loss.item():.4f}")

## 4. DDIM: Accelerated Sampling

**Problem**: DDPM requires 1000 steps for sampling (slow!)

**Solution**: DDIM uses deterministic sampling with fewer steps

**Key difference**:
- DDPM: Stochastic (adds noise at each step)
- DDIM: Deterministic (direct path to x₀)

Can sample in ~50 steps instead of 1000!

In [ ]:
class DDIM:
    """
    Denoising Diffusion Implicit Models (faster sampling).
    """
    
    def __init__(self, model: nn.Module, timesteps: int = 1000, 
                 schedule: str = "linear", device: str = "cpu"):
        self.model = model.to(device)
        self.timesteps = timesteps
        self.device = device
        
        # Get noise schedule
        betas, alphas, alphas_cumprod = get_noise_schedule(schedule, timesteps)
        
        self.betas = betas.to(device)
        self.alphas = alphas.to(device)
        self.alphas_cumprod = alphas_cumprod.to(device)
        
        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod).to(device)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod).to(device)
    
    @torch.no_grad()
    def ddim_sample(self, x: torch.Tensor, t: int, t_next: int, eta: float = 0.0) -> torch.Tensor:
        """
        DDIM sampling step.
        
        η=0: Deterministic (fast)
        η=1: Stochastic (same as DDPM)
        
        Args:
            x: (B, C, H, W) images at step t
            t: current timestep
            t_next: next timestep
            eta: stochasticity parameter
            
        Returns:
            x_next: (B, C, H, W) images at step t_next
        """
        batch_size = x.shape[0]
        t_tensor = torch.full((batch_size,), t, device=self.device, dtype=torch.long)
        
        # Predict noise
        noise_pred = self.model(x, t_tensor)
        
        # Get alpha terms
        alpha_bar_t = self.alphas_cumprod[t]
        alpha_bar_t_next = self.alphas_cumprod[t_next] if t_next >= 0 else torch.tensor(1.0)
        
        # Predict x_0
        x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * noise_pred) / torch.sqrt(alpha_bar_t)
        
        # Clip x_0 prediction
        x0_pred = torch.clamp(x0_pred, -1.0, 1.0)
        
        # Compute direction pointing to x_t
        direction = torch.sqrt(1 - alpha_bar_t_next - eta**2 * (1 - alpha_bar_t_next) / (1 - alpha_bar_t) * (1 - alpha_bar_t / alpha_bar_t_next)) * noise_pred
        
        # Compute x_{t-1}
        x_next = torch.sqrt(alpha_bar_t_next) * x0_pred + direction
        
        if eta > 0:
            noise = torch.randn_like(x)
            variance = eta * torch.sqrt((1 - alpha_bar_t_next) / (1 - alpha_bar_t)) * torch.sqrt(1 - alpha_bar_t / alpha_bar_t_next)
            x_next = x_next + variance * noise
        
        return x_next
    
    @torch.no_grad()
    def sample(self, shape: Tuple[int, ...], num_steps: int = 50, eta: float = 0.0) -> torch.Tensor:
        """
        Generate samples with DDIM (fewer steps).
        
        Args:
            shape: (batch_size, channels, height, width)
            num_steps: Number of sampling steps (much less than training steps!)
            eta: Stochasticity (0=fully deterministic)
            
        Returns:
            samples: (batch_size, channels, height, width)
        """
        self.model.eval()
        
        # Create subset of timesteps
        step_size = self.timesteps // num_steps
        timesteps = list(range(0, self.timesteps, step_size))
        timesteps = timesteps[:num_steps]
        timesteps = timesteps[::-1]  # Reverse order
        
        # Start from pure noise
        x = torch.randn(shape, device=self.device)
        
        # DDIM sampling
        for i, t in enumerate(timesteps):
            t_next = timesteps[i + 1] if i < len(timesteps) - 1 else -1
            x = self.ddim_sample(x, t, t_next, eta)
        
        return x

# Compare DDPM and DDIM
print("DDPM vs DDIM:")
print("="*70)

comparison = [
    ["Aspect", "DDPM", "DDIM"],
    ["Sampling", "Stochastic", "Deterministic (η=0)"],
    ["Steps needed", "~1000", "~50"],
    ["Speed", "Slow", "20x faster"],
    ["Quality", "High", "Similar"],
    ["Training", "Same", "Same"],
    ["Flexibility", "Fixed steps", "Variable steps"]
]

for row in comparison:
    print(f"{row[0]:<20} {row[1]:<25} {row[2]:<25}")

# Visualize sampling trajectories
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# DDPM trajectory (stochastic)
ax1 = axes[0]
ax1.set_title('DDPM Sampling (1000 steps)', fontsize=12, weight='bold')
ax1.set_xlabel('Step', fontsize=10)
ax1.set_ylabel('Signal Strength', fontsize=10)

steps_ddpm = np.arange(1000)
_, _, alphas_cumprod = get_noise_schedule("cosine", 1000)
signal = alphas_cumprod.numpy()

# Add randomness to show stochasticity
trajectory_ddpm = signal + np.random.randn(1000) * 0.02
trajectory_ddpm = np.clip(trajectory_ddpm, 0, 1)

ax1.plot(steps_ddpm, trajectory_ddpm, 'b-', linewidth=2, alpha=0.7, label='Sampling path')
ax1.plot(steps_ddpm, signal, 'r--', linewidth=1, alpha=0.5, label='Expected')
ax1.legend()
ax1.grid(True, alpha=0.3)

# DDIM trajectory (deterministic)
ax2 = axes[1]
ax2.set_title('DDIM Sampling (50 steps)', fontsize=12, weight='bold')
ax2.set_xlabel('Step', fontsize=10)
ax2.set_ylabel('Signal Strength', fontsize=10)

num_steps_ddim = 50
step_indices = np.linspace(0, 999, num_steps_ddim, dtype=int)
steps_ddim = np.arange(num_steps_ddim)
signal_ddim = signal[step_indices]

ax2.plot(steps_ddim, signal_ddim, 'g-', linewidth=2, marker='o', 
        markersize=4, label='Sampling path')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight:")
print("  DDIM can skip timesteps without loss of quality!")
print("  This makes real-time generation possible")

## 5. Stable Diffusion

**Stable Diffusion** = Latent Diffusion Model

Instead of operating on pixels, works in **latent space**:

```
Image → VAE Encoder →Latent → Diffusion → VAE Decoder → Image
```

**Components**:
1. **VAE Encoder/Decoder**: Compress images to latent space
2. **U-Net**: Denoise latent representations
3. **CLIP Text Encoder**: Encode text prompts
4. **Cross-Attention**: Condition on text

**Advantage**: 8x8 smaller latent space → 64x faster!

In [ ]:
def visualize_stable_diffusion_architecture():
    """Visualize Stable Diffusion components."""
    
    fig, ax = plt.subplots(1, 1, figsize=(16, 10))
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    
    # Title
    ax.text(0.5, 0.95, 'Stable Diffusion Architecture', ha='center', 
           fontsize=16, weight='bold')
    
    # Components
    components = [
        # Encoding path
        (0.05, 0.7, 0.12, 0.12, "Input\nImage\n512×512×3", 'lightblue'),
        (0.22, 0.7, 0.12, 0.12, "VAE\nEncoder\n↓8x", 'lightgreen'),
        (0.39, 0.7, 0.12, 0.12, "Latent\n64×64×4", 'lightyellow'),
        
        # Text path
        (0.05, 0.5, 0.12, 0.12, "Text\nPrompt", 'lightcoral'),
        (0.22, 0.5, 0.12, 0.12, "CLIP\nEncoder", 'lightpink'),
        
        # Diffusion process
        (0.56, 0.7, 0.18, 0.12, "U-Net\n+ Cross-Attention\n(denoising)", 'lightsteelblue'),
        
        # Decoding path
        (0.79, 0.7, 0.12, 0.12, "VAE\nDecoder\n↑8x", 'lightgreen'),
        (0.92, 0.7, 0.045, 0.12, "Out", 'lightblue'),
    ]
    
    for x, y, w, h, label, color in components:
        rect = plt.Rectangle((x, y), w, h, color=color, ec='black', linewidth=2)
        ax.add_patch(rect)
        lines = label.split('\n')
        for i, line in enumerate(lines):
            offset = (len(lines) - 1) * 0.02
            ax.text(x + w/2, y + h/2 + offset - i*0.04, line, 
                   ha='center', va='center', fontsize=9, weight='bold')
    
    # Arrows
    arrows = [
        # Encoding
        ((0.17, 0.76), (0.22, 0.76), 'black'),
        ((0.34, 0.76), (0.39, 0.76), 'black'),
        # Text
        ((0.17, 0.56), (0.22, 0.56), 'black'),
        ((0.34, 0.56), (0.39, 0.65), 'red'),  # Text to U-Net
        # Latent to U-Net
        ((0.51, 0.76), (0.56, 0.76), 'black'),
        # U-Net to decoder
        ((0.74, 0.76), (0.79, 0.76), 'black'),
        # Decoder to output
        ((0.91, 0.76), (0.92, 0.76), 'black'),
    ]
    
    for start, end, color in arrows:
        ax.annotate('', xy=end, xytext=start,
                   arrowprops=dict(arrowstyle='->', lw=2, color=color))
    
    # Labels
    ax.text(0.28, 0.84, 'Compress', ha='center', fontsize=9, 
           style='italic', color='green')
    ax.text(0.65, 0.84, 'Denoise (T steps)', ha='center', fontsize=9, 
           style='italic', color='blue')
    ax.text(0.855, 0.84, 'Decompress', ha='center', fontsize=9, 
           style='italic', color='green')
    ax.text(0.455, 0.62, 'Condition', ha='center', fontsize=9, 
           style='italic', color='red')
    
    # Details boxes
    details = [
        (0.15, 0.35, 0.25, 0.25, "VAE (Encoder/Decoder)", 
         ["• Compresses 512² → 64²", "• Latent dim: 4", "• 8x8 smaller!", 
          "• KL regularization", "• Trained separately"]),
        
        (0.45, 0.35, 0.25, 0.25, "U-Net + Attention",
         ["• ResNet blocks", "• Self-attention layers", "• Cross-attention (text)", 
          "• Time embedding", "• Skip connections"]),
        
        (0.75, 0.35, 0.2, 0.25, "Advantages",
         ["• 64x faster than pixel", "• Lower memory", "• Better quality", 
          "• Text conditioning"])
    ]
    
    for x, y, w, h, title, points in details:
        rect = plt.Rectangle((x, y), w, h, color='white', ec='black', 
                            linewidth=2, linestyle='--')
        ax.add_patch(rect)
        
        ax.text(x + w/2, y + h - 0.03, title, ha='center', 
               fontsize=10, weight='bold')
        
        for i, point in enumerate(points):
            ax.text(x + 0.02, y + h - 0.08 - i*0.04, point, 
                   fontsize=8, va='top')
    
    # Process flow annotation
    ax.text(0.5, 0.15, 'Diffusion Process: noise → latent → denoise → decode → image', 
           ha='center', fontsize=11, style='italic',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    print("\nStable Diffusion Details:")
    print("="*70)
    
    print("\n1. LATENT SPACE")
    print("  • Image: 512×512×3 = 786,432 dimensions")
    print("  • Latent: 64×64×4 = 16,384 dimensions")
    print("  • Compression: 48x smaller!")
    print("  • Speed: ~64x faster diffusion")
    
    print("\n2. TEXT CONDITIONING")
    print("  • CLIP text encoder converts prompt to embeddings")
    print("  • Cross-attention injects text information")
    print("  • Each U-Net layer attends to text")
    print("  • Enables text-to-image generation")
    
    print("\n3. TRAINING STAGES")
    print("  • Stage 1: Train VAE on images")
    print("  • Stage 2: Train U-Net on latents")
    print("  • Only need to train U-Net for conditioning!")
    
    print("\n4. INFERENCE")
    print("  • Start with random latent noise")
    print("  • Denoise with text guidance (DDIM ~50 steps)")
    print("  • Decode latent to image")
    print("  • Total time: ~2-3 seconds on GPU")

visualize_stable_diffusion_architecture()

# Cross-attention mechanism
class CrossAttention(nn.Module):
    """
    Cross-attention for text conditioning.
    """
    
    def __init__(self, query_dim: int, context_dim: int, num_heads: int = 8):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = query_dim // num_heads
        
        self.to_q = nn.Linear(query_dim, query_dim)
        self.to_k = nn.Linear(context_dim, query_dim)
        self.to_v = nn.Linear(context_dim, query_dim)
        self.to_out = nn.Linear(query_dim, query_dim)
        
    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, query_dim) - image features
            context: (batch, context_len, context_dim) - text features
            
        Returns:
            out: (batch, seq_len, query_dim) - attended features
        """
        batch_size = x.shape[0]
        
        # Project to Q, K, V
        q = self.to_q(x)  # (B, seq_len, query_dim)
        k = self.to_k(context)  # (B, context_len, query_dim)
        v = self.to_v(context)  # (B, context_len, query_dim)
        
        # Reshape for multi-head attention
        q = q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = F.softmax(scores, dim=-1)
        
        out = torch.matmul(attn, v)
        
        # Reshape back
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.head_dim)
        
        return self.to_out(out)

# Test cross-attention
cross_attn = CrossAttention(query_dim=512, context_dim=768, num_heads=8).to(device)

batch = 2
seq_len = 64 * 64  # Flattened 64x64 latent
context_len = 77  # CLIP text sequence length

test_image_features = torch.randn(batch, seq_len, 512).to(device)
test_text_features = torch.randn(batch, context_len, 768).to(device)

with torch.no_grad():
    attended = cross_attn(test_image_features, test_text_features)

print("\nCross-Attention Test:")
print(f"  Image features: {test_image_features.shape}")
print(f"  Text features: {test_text_features.shape}")
print(f"  Attended features: {attended.shape}")
print("  ✓ Text information injected into image features!")

## 6. ControlNet

**ControlNet** adds spatial control to Stable Diffusion

**Idea**: Copy U-Net, freeze original, train copy with control input

**Control signals**:
- Canny edges
- Depth maps
- Pose keypoints
- Segmentation masks

Enables precise control over generation!

In [ ]:
def visualize_controlnet():
    """Visualize ControlNet architecture."""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # 1. Architecture diagram
    ax1 = axes[0, 0]
    ax1.axis('off')
    ax1.set_title('ControlNet Architecture', fontsize=12, weight='bold')
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    
    # Original U-Net
    rect1 = plt.Rectangle((0.1, 0.6), 0.3, 0.25, color='lightblue', 
                         ec='black', linewidth=2)
    ax1.add_patch(rect1)
    ax1.text(0.25, 0.725, 'Original U-Net\n(Frozen)', ha='center', 
            va='center', fontsize=9, weight='bold')
    
    # ControlNet copy
    rect2 = plt.Rectangle((0.1, 0.2), 0.3, 0.25, color='lightgreen', 
                         ec='black', linewidth=2)
    ax1.add_patch(rect2)
    ax1.text(0.25, 0.325, 'ControlNet Copy\n(Trainable)', ha='center', 
            va='center', fontsize=9, weight='bold')
    
    # Control input
    rect3 = plt.Rectangle((0.1, 0.05), 0.15, 0.1, color='lightyellow', 
                         ec='black', linewidth=2)
    ax1.add_patch(rect3)
    ax1.text(0.175, 0.1, 'Control\nInput', ha='center', va='center', 
            fontsize=8, weight='bold')
    
    # Arrows
    ax1.annotate('', xy=(0.25, 0.2), xytext=(0.175, 0.15),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    ax1.annotate('', xy=(0.25, 0.6), xytext=(0.25, 0.47),
                arrowprops=dict(arrowstyle='->', lw=2, color='red'))
    ax1.text(0.3, 0.53, 'Add\nresidual', fontsize=7, color='red')
    
    # Output
    rect4 = plt.Rectangle((0.55, 0.6), 0.3, 0.25, color='lightcoral', 
                         ec='black', linewidth=2)
    ax1.add_patch(rect4)
    ax1.text(0.7, 0.725, 'Output\n(Controlled)', ha='center', 
            va='center', fontsize=9, weight='bold')
    
    ax1.annotate('', xy=(0.55, 0.725), xytext=(0.42, 0.725),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    
    # 2. Control types
    ax2 = axes[0, 1]
    ax2.axis('off')
    ax2.set_title('Control Signal Types', fontsize=12, weight='bold')
    
    control_types = [
        (0.15, 'Canny Edge', 'Edge detection\nof reference'),
        (0.35, 'Depth Map', 'Depth information\nfor 3D structure'),
        (0.55, 'Pose', 'Human keypoints\nfor pose control'),
        (0.75, 'Segmentation', 'Semantic masks\nfor layout')
    ]
    
    for y, name, desc in control_types:
        rect = plt.Rectangle((0.1, y), 0.8, 0.12, color='lightsteelblue', 
                            ec='black', linewidth=1.5)
        ax2.add_patch(rect)
        ax2.text(0.2, y + 0.06, name, fontsize=10, weight='bold', va='center')
        lines = desc.split('\n')
        for i, line in enumerate(lines):
            ax2.text(0.5, y + 0.08 - i*0.03, line, fontsize=8, va='center')
    
    # 3. Training process
    ax3 = axes[1, 0]
    ax3.set_title('Training Stages', fontsize=12, weight='bold')
    ax3.set_xlabel('Training Steps', fontsize=10)
    ax3.set_ylabel('Loss', fontsize=10)
    
    steps = np.arange(0, 100)
    
    # Frozen U-Net (flat)
    unet_loss = np.ones(100) * 0.5
    ax3.plot(steps, unet_loss, 'b--', linewidth=2, label='U-Net (frozen)', alpha=0.7)
    
    # ControlNet (decreasing)
    control_loss = 2.0 * np.exp(-steps / 30) + 0.3 + np.random.randn(100) * 0.05
    control_loss = np.clip(control_loss, 0.2, 2.5)
    ax3.plot(steps, control_loss, 'g-', linewidth=2, label='ControlNet (training)')
    
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(0, 2.5)
    
    # 4. Comparison table
    ax4 = axes[1, 1]
    ax4.axis('off')
    ax4.set_title('With vs Without ControlNet', fontsize=12, weight='bold')
    
    comparison = [
        ['Aspect', 'Without Control', 'With ControlNet'],
        ['Precision', 'Low', 'High'],
        ['Consistency', 'Variable', 'Consistent'],
        ['Pose control', 'No', 'Yes'],
        ['Layout control', 'No', 'Yes'],
        ['Training', 'N/A', 'Extra model'],
        ['Speed', 'Fast', 'Slightly slower']
    ]
    
    for i, row in enumerate(comparison):
        y_pos = 0.85 - i * 0.12
        colors = ['black', 'red', 'green']
        for j, (cell, color) in enumerate(zip(row, colors)):
            x_pos = 0.05 + j * 0.3
            weight = 'bold' if i == 0 else 'normal'
            ax4.text(x_pos, y_pos, cell, fontsize=9, weight=weight,
                    color=color if i > 0 else 'black')
        
        if i == 0:
            ax4.axhline(y=y_pos-0.05, xmin=0.05, xmax=0.95,
                       color='black', linewidth=1)
    
    plt.tight_layout()
    plt.show()
    
    print("\nControlNet:")
    print("="*70)
    
    print("\nKEY IDEA:")
    print("  • Copy entire U-Net architecture")
    print("  • Freeze original weights")
    print("  • Train copy to process control signal")
    print("  • Add outputs via zero-initialized convolutions")
    
    print("\nTRAINING:")
    print("  • Original: 50% randomly drop text conditioning")
    print("  • ControlNet: Always use control signal")
    print("  • Loss: Same diffusion objective")
    print("  • Result: Precise spatial control!")
    
    print("\nAPPLICATIONS:")
    print("  • Pose-controlled character generation")
    print("  • Depth-guided landscape creation")
    print("  • Edge-based line art colorization")
    print("  • Layout-constrained image synthesis")

visualize_controlnet()

class ZeroConv(nn.Module):
    """
    Zero-initialized convolution for ControlNet.
    
    Starts with zero output, gradually learns contribution.
    """
    
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, 1)
        nn.init.zeros_(self.conv.weight)
        nn.init.zeros_(self.conv.bias)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)

print("\n" + "="*70)
print("ZERO CONVOLUTION")
print("="*70)
print("\nWhy zero initialization?")
print("  • At start, ControlNet contributes nothing")
print("  • Original model still works perfectly")
print("  • Gradually learns to add control information")
print("  • No harmful effects during early training")

# Test zero conv
zero_conv = ZeroConv(64, 64).to(device)
test_input = torch.randn(1, 64, 32, 32).to(device)

with torch.no_grad():
    output = zero_conv(test_input)

print(f"\nTest:")
print(f"  Input: {test_input.shape}")
print(f"  Output: {output.shape}")
print(f"  Output sum: {output.sum().item():.10f} (should be ~0)")

## 7. LoRA for Diffusion Models

**LoRA** (Low-Rank Adaptation) for efficient fine-tuning

**Idea**: Instead of updating all weights, add low-rank matrices

$$W' = W + \Delta W = W + BA$$

Where:
- $W$: Original weights (frozen)
- $B \in \mathbb{R}^{d \times r}$: Down-projection
- $A \in \mathbb{R}^{r \times k}$: Up-projection
- $r \ll \min(d, k)$: Rank (typically 4-16)

**Advantages**:
- Train 0.1% of parameters
- Merge with original model
- Fast switching between styles

In [ ]:
class LoRALayer(nn.Module):
    """
    LoRA layer for efficient fine-tuning.
    """
    
    def __init__(self, original_layer: nn.Module, rank: int = 4, alpha: float = 1.0):
        super().__init__()
        
        self.original_layer = original_layer
        self.rank = rank
        self.alpha = alpha
        
        # Get dimensions
        if isinstance(original_layer, nn.Linear):
            d_out, d_in = original_layer.weight.shape
        elif isinstance(original_layer, nn.Conv2d):
            d_out = original_layer.out_channels
            d_in = original_layer.in_channels * original_layer.kernel_size[0] * original_layer.kernel_size[1]
        else:
            raise ValueError(f"Unsupported layer type: {type(original_layer)}")
        
        # Low-rank matrices
        self.lora_A = nn.Parameter(torch.randn(rank, d_in) / rank)
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))
        
        # Freeze original
        for param in self.original_layer.parameters():
            param.requires_grad = False
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward with LoRA adaptation.
        
        out = W·x + (B·A)·x
        """
        # Original output
        original_out = self.original_layer(x)
        
        # LoRA adaptation
        if isinstance(self.original_layer, nn.Linear):
            lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.alpha
        elif isinstance(self.original_layer, nn.Conv2d):
            # For Conv2d, apply as linear transformation
            batch, channels, height, width = x.shape
            x_flat = x.permute(0, 2, 3, 1).reshape(-1, channels)
            lora_out = (x_flat @ self.lora_A.T @ self.lora_B.T) * self.alpha
            lora_out = lora_out.reshape(batch, height, width, -1).permute(0, 3, 1, 2)
        
        return original_out + lora_out
    
    def merge_weights(self):
        """
        Merge LoRA weights into original layer (for inference).
        """
        with torch.no_grad():
            delta_W = self.lora_B @ self.lora_A * self.alpha
            
            if isinstance(self.original_layer, nn.Linear):
                self.original_layer.weight.data += delta_W
            elif isinstance(self.original_layer, nn.Conv2d):
                # Reshape for conv weights
                delta_W = delta_W.reshape(
                    self.original_layer.out_channels,
                    self.original_layer.in_channels,
                    self.original_layer.kernel_size[0],
                    self.original_layer.kernel_size[1]
                )
                self.original_layer.weight.data += delta_W

# Compare parameter counts
original_linear = nn.Linear(1024, 1024).to(device)
lora_linear = LoRALayer(original_linear, rank=8, alpha=1.0).to(device)

original_params = sum(p.numel() for p in original_linear.parameters() if p.requires_grad)
lora_params = sum(p.numel() for p in lora_linear.parameters() if p.requires_grad)

print("LoRA Parameter Efficiency:")
print("="*70)
print(f"\nOriginal layer: {original_params:,} trainable parameters")
print(f"LoRA adaptation: {lora_params:,} trainable parameters")
print(f"Reduction: {(1 - lora_params/original_params)*100:.2f}%")
print(f"\nFor full Stable Diffusion U-Net:")
print(f"  Original: ~860M parameters")
print(f"  LoRA (rank=8): ~1M parameters (0.12%)")
print(f"  File size: ~3.5MB vs 3.5GB (1000x smaller!)")

# Visualize LoRA
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Matrix decomposition
ax1 = axes[0]
ax1.axis('off')
ax1.set_title('LoRA: Low-Rank Adaptation', fontsize=12, weight='bold')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

# Original weight matrix
rect_W = plt.Rectangle((0.05, 0.5), 0.15, 0.3, color='lightblue', 
                       ec='black', linewidth=2)
ax1.add_patch(rect_W)
ax1.text(0.125, 0.42, 'W\n1024×1024\nFrozen', ha='center', fontsize=9, weight='bold')

# Plus sign
ax1.text(0.25, 0.65, '+', fontsize=20, weight='bold', ha='center')

# Delta W = B·A
rect_B = plt.Rectangle((0.35, 0.5), 0.15, 0.3, color='lightcoral', 
                       ec='black', linewidth=2)
ax1.add_patch(rect_B)
ax1.text(0.425, 0.42, 'B\n1024×8', ha='center', fontsize=9, weight='bold')

ax1.text(0.53, 0.65, '·', fontsize=20, weight='bold', ha='center')

rect_A = plt.Rectangle((0.58, 0.6), 0.08, 0.15, color='lightgreen', 
                       ec='black', linewidth=2)
ax1.add_patch(rect_A)
ax1.text(0.62, 0.54, 'A\n8×1024', ha='center', fontsize=9, weight='bold')

# Equals
ax1.text(0.7, 0.65, '=', fontsize=20, weight='bold', ha='center')

# Result
rect_result = plt.Rectangle((0.8, 0.5), 0.15, 0.3, color='lightyellow', 
                           ec='black', linewidth=2)
ax1.add_patch(rect_result)
ax1.text(0.875, 0.42, "W'\n1024×1024\nAdapted", ha='center', fontsize=9, weight='bold')

# Annotation
ax1.text(0.5, 0.25, 'Only train B and A (16K params instead of 1M!)', 
        ha='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 2. Training efficiency
ax2 = axes[1]
ax2.set_title('Training Efficiency', fontsize=12, weight='bold')

metrics = ['Parameters', 'Memory', 'Training Time', 'File Size']
full_ft = [100, 100, 100, 100]
lora_ft = [0.12, 20, 30, 0.1]

x = np.arange(len(metrics))
width = 0.35

ax2.bar(x - width/2, full_ft, width, label='Full Fine-tune', color='lightblue')
ax2.bar(x + width/2, lora_ft, width, label='LoRA', color='lightgreen')

ax2.set_ylabel('Relative Cost (%)')
ax2.set_xticks(x)
ax2.set_xticklabels(metrics, rotation=45, ha='right')
ax2.legend()
ax2.grid(True, axis='y', alpha=0.3)
ax2.set_ylim(0, 110)

# Add value labels
for i, (ff, lf) in enumerate(zip(full_ft, lora_ft)):
    ax2.text(i - width/2, ff + 2, f'{ff}%', ha='center', fontsize=9)
    ax2.text(i + width/2, lf + 2, f'{lf}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nLoRA Applications in Stable Diffusion:")
print("-"*70)
print("\n• STYLE TRANSFER: Train LoRA on artist's style")
print("  Example: 'anime style', 'oil painting', 'cyberpunk'")
print("  Training: 1000 images, 2-4 hours on single GPU")

print("\n• CHARACTER CONSISTENCY: Fine-tune on specific character")
print("  Example: Generate same character in different scenes")
print("  Training: 10-20 images, 30 minutes")

print("\n• CONCEPT LEARNING: Teach new concepts")
print("  Example: Rare objects, custom creatures")
print("  Training: 5-15 images, 15 minutes")

print("\n• MULTIPLE LoRAs: Can apply multiple simultaneously")
print("  Example: 'anime style' + 'cyberpunk' + 'character X'")
print("  Just add the delta weights!")

## Summary

You've mastered diffusion models:
- ✅ Forward/reverse diffusion process
- ✅ Noise schedules (linear, cosine, quadratic)
- ✅ DDPM training and sampling
- ✅ DDIM accelerated sampling (20x faster)
- ✅ Stable Diffusion architecture (VAE + U-Net + CLIP)
- ✅ Cross-attention for text conditioning
- ✅ ControlNet for spatial control
- ✅ LoRA for efficient fine-tuning

**Key Insights**:
1. **Diffusion** gradually adds noise, then learns to reverse
2. **DDPM** trains on predicting noise, samples iteratively
3. **DDIM** enables deterministic fast sampling (50 steps vs 1000)
4. **Stable Diffusion** works in latent space (64x faster)
5. **ControlNet** adds precise spatial control without retraining base model
6. **LoRA** enables 1000x more efficient fine-tuning

**Comparison with Other Generative Models**:

| Model | Quality | Speed | Training | Control |
|-------|---------|-------|----------|----------|
| VAE | Medium | Fast | Stable | Low |
| GAN | High | Fast | Unstable | Low |
| Diffusion | Highest | Slow* | Stable | High |

*With DDIM and latent space, now competitive!

**State-of-the-Art Models** (2024):
- **SDXL**: Stable Diffusion XL (better quality, 1024×1024)
- **Imagen**: Google's text-to-image (photorealistic)
- **DALL-E 3**: OpenAI (excellent prompt following)
- **Midjourney V6**: Best artistic quality
- **Stable Video Diffusion**: Text-to-video

**Applications**:
- Text-to-image generation
- Image editing (inpainting, outpainting)
- Super-resolution
- Style transfer
- Video generation
- 3D generation (Point-E, Shap-E)
- Audio generation (AudioLDM)

**Next Steps**:
- Try Stable Diffusion with HuggingFace Diffusers
- Train LoRA on custom dataset
- Experiment with ControlNet
- Explore video diffusion models

## Exercises

1. **Implement DDPM**: Train on MNIST from scratch  
2. **Compare schedules**: Test linear vs cosine noise schedules
3. **DDIM speedup**: Measure quality vs speed tradeoff
4. **Train LoRA**: Fine-tune Stable Diffusion on custom style
5. **ControlNet**: Generate images with pose/edge control